## Set current dir to project root dir

In [1]:
path <- getwd()
markers <- c(".git", "Makefile", "renv.lock", ".Rprofile")
while (!any(file.exists(file.path(path, markers)))) {
  parent <- dirname(path)
  if (parent == path) stop("Could not find project root")
  path <- parent
}
setwd(path)
rm(path, parent, markers)

## Activate project's local R env. using renv

In [2]:
if (file.exists("renv/activate.R")) {
  source("renv/activate.R")
} else {
  stop("Could not find renv/activate.R in project root")
}

# Imports

In [3]:
library(eegUtils)
library(readr)


Attaching package: ‘eegUtils’


The following object is masked from ‘package:stats’:

    filter




# Functions

In [4]:
get_subject_folders <- function(parent_directory) {
  folders <- list.dirs(parent_directory, full.names = TRUE, recursive = FALSE)
  folders <- folders[grepl("/sub-", folders)]
  sort(folders)
}

In [5]:
extract_unique_stimuli <- function(file_path) {
  # Parses a BrainVision .vmrk file and returns a sorted list
  # of all unique stimulus descriptions (trigger codes).
  stimuli <- c()
  lines <- readLines(file_path, encoding = "UTF-8")
  for (line in lines) {
    if (grepl("^Mk", line)) {
      tryCatch({
        content <- strsplit(line, "=")[[1]][2]
        parts <- strsplit(content, ",")[[1]]
        marker_type <- trimws(parts[1])
        description <- trimws(parts[2])
        if (marker_type == "Stimulus") {
          stimuli <- c(stimuli, description)
        }
      }, error = function(e) NULL)
    }
  }
  sort(unique(stimuli))
}

# Variables

In [6]:
main_data_folder <- "./ds006018"
tasks <- c("task-auditoryoddball", "task-flanker",
           "task-visualoddball", "task-visualsearch")

# Main

In [ ]:
subject_folders <- get_subject_folders(main_data_folder)

for (subject_path in subject_folders) {
  sub_number <- basename(subject_path)
  out_dir <- file.path("./ds006018_per_stimuli", sub_number)
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

  for (task in tasks) {
    path_to_vhdr <- file.path(subject_path, "eeg",
                              paste0(sub_number, "_", task, "_eeg.vhdr"))
    path_to_vmrk <- file.path(subject_path, "eeg",
                              paste0(sub_number, "_", task, "_eeg.vmrk"))

    if (!file.exists(path_to_vhdr)) {
      cat(path_to_vhdr, "File not found.\n")
      next
    }

    cat("The file exists!\n")

    # Get unique stimulus codes from .vmrk
    unique_stimuli <- extract_unique_stimuli(path_to_vmrk)
    cat(unique_stimuli, "\n")

    # 1. Load BrainVision data
    raw <- import_raw(path_to_vhdr)

    # 2. Set standard 10-20 montage
    raw <- electrode_locations(raw, montage = "standard_1020",
                               overwrite = TRUE)

    # 3. Bandpass filter 0.1-40 Hz
    raw <- eeg_filter(raw, low_freq = 0.1, high_freq = 40.0,
                      method = "iir")

    # 4. Epoch per stimulus, baseline correct, export
    for (stimulus in unique_stimuli) {
      clean_name <- gsub("[/ ]", "", stimulus)
      clean_name <- paste0("Stimulus_", clean_name)

      tryCatch({
        epochs <- epoch_data(raw,
                             events = stimulus,
                             epoch_start = -0.2,
                             epoch_end  =  0.8)

        # 5. Baseline correction [-0.2, 0.0]
        epochs <- rm_baseline(epochs, baseline = c(-0.2, 0))

        n_epochs <- length(unique(epochs$signals$epoch))

        # 6. Export to CSV
        df <- as.data.frame(epochs)
        out_path <- file.path(out_dir,
                              paste0(task, "_", clean_name, ".csv"))
        write_csv(df, out_path)
        cat(sprintf("Successfully created: eeg_%s.csv (%d trials)\n",
                    clean_name, n_epochs))
      }, error = function(e) {
        cat(sprintf("Skipping %s: %s\n", stimulus, conditionMessage(e)))
      })
    }
  }
  
  break  # DEBUG: remove this break to process all subjects
}

The file exists!
S  1 S 70 S 80 S180 


Importing Brain Vision Analyzer file ./ds006018/sub-001/eeg/sub-001_task-auditoryoddball_eeg.vhdr



ERROR: Error in montage_check(montage): Unknown montage specified.
